In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Identifying Duplicated Species

In [41]:
orthoDist = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet")
display(orthoDist)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC
0,ENSG00000000003,ENSMUSG00000067377,ortholog_one2one,100.0,44.054842,0.001212,4.687707,1.130272,0.0000
1,ENSG00000000005,ENSMUSG00000031250,ortholog_one2one,100.0,5.172072,0.000097,1.969849,1.141118,0.1250
2,ENSG00000000419,ENSMUSG00000078919,ortholog_one2one,100.0,188.287732,0.003195,2.569262,1.211158,0.0000
3,ENSG00000000457,ENSMUSG00000026584,ortholog_one2one,100.0,37.114733,0.001000,2.030044,0.814870,0.0000
4,ENSG00000000460,ENSMUSG00000041406,ortholog_one2one,0.0,9.113600,0.000181,1.989693,0.959842,0.1875
...,...,...,...,...,...,...,...,...,...
28009,ENSG00000310576,ENSMUSG00000035595,ortholog_one2one,100.0,NaN,NaN,NaN,NaN,NaN
28010,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28011,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28012,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# Identifies the number of mouse genes associated with each human gene.
groupedHumanGene = orthoDist.groupby("Gene stable ID")["Mouse gene stable ID"].apply(list).reset_index(name="Mouse gene stable ID")

# Adds in the homology type to the dataframe.
groupedHumanGene = groupedHumanGene.merge(orthoDist.loc[:, ["Gene stable ID", "Mouse homology type"]])

# Identifies the human genes that have more than one mouse gene associated with them.
groupedHumanGene["Num Mouse Dupes"] = groupedHumanGene["Mouse gene stable ID"].apply(len)

# Creates a new column that identifies which one-to-many gene experienced a duplication event in humans. 
groupedHumanGene["Duplicated Species"] = np.where((groupedHumanGene["Gene stable ID"].isin(groupedHumanGene[groupedHumanGene["Num Mouse Dupes"] > 1]["Gene stable ID"])) & (groupedHumanGene["Mouse homology type"] == "ortholog_one2many"), "Mouse", "NA")

In [43]:
groupedHumanGene

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Num Mouse Dupes,Duplicated Species
0,ENSG00000000003,[ENSMUSG00000067377],ortholog_one2one,1,NA
1,ENSG00000000005,[ENSMUSG00000031250],ortholog_one2one,1,NA
2,ENSG00000000419,[ENSMUSG00000078919],ortholog_one2one,1,NA
3,ENSG00000000457,[ENSMUSG00000026584],ortholog_one2one,1,NA
4,ENSG00000000460,[ENSMUSG00000041406],ortholog_one2one,1,NA
...,...,...,...,...,...
28009,ENSG00000310576,[ENSMUSG00000035595],ortholog_one2one,1,NA
28010,ENSG00000310579,[nan],NaN,1,NA
28011,ENSG00000310583,[nan],NaN,1,NA
28012,ENSG00000310590,[nan],NaN,1,NA


In [44]:
# Transfers the information above to the orthologTableDist dataframe.
orthoDistHumanDup = orthoDist.merge(groupedHumanGene.loc[:, ["Gene stable ID", "Duplicated Species", "Num Mouse Dupes"]], left_on="Gene stable ID", right_on="Gene stable ID", how="outer")

In [45]:
# Same steps as three cells above, but for mouse genes.
groupedMouseGene = orthoDist.groupby("Mouse gene stable ID")["Gene stable ID"].apply(list).reset_index(name="Gene stable ID")
groupedMouseGene = groupedMouseGene.merge(orthoDist.loc[:, ["Mouse gene stable ID", "Mouse homology type"]])
groupedMouseGene["Num Human Dupes"] = groupedMouseGene["Gene stable ID"].apply(len)
groupedMouseGene["Duplicated Species"] = np.where((groupedMouseGene["Mouse gene stable ID"].isin(groupedMouseGene[groupedMouseGene["Num Human Dupes"] > 1]["Mouse gene stable ID"])) & (groupedMouseGene["Mouse homology type"] == "ortholog_one2many"), "Human", "NA")

In [46]:
groupedMouseGene

,Mouse gene stable ID,Gene stable ID,Mouse homology type,Num Human Dupes,Duplicated Species
0,ENSMUSG00000000001,[ENSG00000065135],ortholog_one2one,1,NA
1,ENSMUSG00000000028,[ENSG00000093009],ortholog_one2one,1,NA
2,ENSMUSG00000000037,[ENSG00000102098],ortholog_one2one,1,NA
3,ENSMUSG00000000049,[ENSG00000091583],ortholog_one2one,1,NA
4,ENSMUSG00000000056,[ENSG00000141562],ortholog_one2one,1,NA
...,...,...,...,...,...
22464,ENSMUSG00000144248,[ENSG00000289360],ortholog_one2one,1,NA
22465,ENSMUSG00000144259,[ENSG00000288706],ortholog_one2one,1,NA
22466,ENSMUSG00000144287,[ENSG00000255154],ortholog_one2one,1,NA
22467,ENSMUSG00001074846,[ENSG00000229972],ortholog_one2one,1,NA


In [47]:
# Transfers the information above to the orthologDistTable dataframe.
orthoDistHumanMouseDup = orthoDistHumanDup.merge(groupedMouseGene.loc[:, ["Mouse gene stable ID", "Duplicated Species", "Num Human Dupes"]], on="Mouse gene stable ID", how="outer")

# Because there are two separate "Duplicated Species" columns, I am going to combine their information into one.
# If the human "Duplicated Species" column is "NA" then we use the mouse "Duplicated Species" column. It will either be "Mouse" or stay "NA."
orthoDistHumanMouseDup["Duplicated Species"] = np.where(orthoDistHumanMouseDup["Duplicated Species_x"] == "NA", orthoDistHumanMouseDup["Duplicated Species_y"], orthoDistHumanMouseDup["Duplicated Species_x"])

# Drop the two separate "Duplicated Species" columns.
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, ~orthoDistHumanMouseDup.columns.isin(["Duplicated Species_x", "Duplicated Species_y"])].drop_duplicates()
display(orthoDistHumanMouseDup)

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Mouse Dupes,Num Human Dupes,Duplicated Species
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,0.004396,2.312719,0.151442,0.0000,1,1.0,NA
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,0.000412,3.188187,1.001638,0.1875,1,1.0,NA
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,0.000073,1.435472,0.161786,0.0625,1,1.0,NA
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,0.002976,4.951117,0.000003,0.2500,1,1.0,NA
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,0.001784,2.933593,0.661562,0.0000,1,1.0,NA
...,...,...,...,...,...,...,...,...,...,...,...,...
3884923,ENSG00000310562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884924,ENSG00000310579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884925,ENSG00000310583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN
3884926,ENSG00000310590,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN


In [48]:
orthoDistHumanMouseDup.columns[-1:]

Index(['Duplicated Species'], dtype='str')

In [49]:
newColOrder = list(orthoDistHumanMouseDup.columns[:-3]) + list(orthoDistHumanMouseDup.columns[-2:-1]) + list(orthoDistHumanMouseDup.columns[-3:-2]) + list(orthoDistHumanMouseDup.columns[-1:])
orthoDistHumanMouseDup = orthoDistHumanMouseDup.loc[:, newColOrder]

In [50]:
orthoDistHumanMouseDup.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.csv", index=False)
orthoDistHumanMouseDup.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupe.parquet", index=False)

In [51]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

In [52]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [53]:
emtabEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSMUSG00000000001,34.911248,129.180584,143.270691,72.898048,72.920467,45.744554,8.901608,54.003405,protein_coding
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.053683,0.000000,0.000000,0.000000,0.000000,protein_coding
ENSMUSG00000000028,2.236124,11.323525,5.613315,22.680435,2.280480,0.871723,0.181636,3.491172,protein_coding
ENSMUSG00000000031,2.567429,1.137826,2650.257043,17.372786,0.758899,1.307234,1.672216,2.287438,lncRNA
ENSMUSG00000000037,1.226929,2.675817,3.285196,0.774847,0.333726,0.011223,0.000000,0.287549,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.332800,0.047076,0.021773,0.015338,0.008382,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109575,1.098742,0.000000,0.000000,0.009073,0.000000,0.000000,0.000000,0.000000,TEC
ENSMUSG00000109576,0.022506,0.000000,0.000000,0.024401,0.012853,0.000000,0.000000,0.000000,TEC


In [54]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDup.merge(gtexEP.loc[:, ["Gene type"]], left_on="Gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Human Gene Type"})

In [55]:
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.merge(emtabEP.loc[:, ["Gene type"]], left_on="Mouse gene stable ID", right_index=True, how="left")
orthoDistHumanMouseDupBioType = orthoDistHumanMouseDupBioType.rename(columns={"Gene type": "Mouse Gene Type"})

In [56]:
orthoDistHumanMouseDupBioType.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.csv")
orthoDistHumanMouseDupBioType.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")

# Statistical Analysis

In [ ]:
data = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDistDupeBioType.parquet")
dataProteinOnly = data[(data["Human Gene Type"] == "protein_coding") & (data["Mouse Gene Type"] == "protein_coding")]

In [24]:
dataProteinOnly

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Num Human Dupes,Num Mouse Dupes,Duplicated Species,Human Gene Type,Mouse Gene Type
0,ENSG00000065135,ENSMUSG00000000001,ortholog_one2one,100.0,197.746445,4.395566e-03,2.312719,0.151442,0.0000,1.0,1,NA,protein_coding,protein_coding
1,ENSG00000093009,ENSMUSG00000000028,ortholog_one2one,100.0,24.533616,4.121514e-04,3.188187,1.001638,0.1875,1.0,1,NA,protein_coding,protein_coding
2,ENSG00000102098,ENSMUSG00000000037,ortholog_one2one,100.0,2.084639,7.261759e-05,1.435472,0.161786,0.0625,1.0,1,NA,protein_coding,protein_coding
3,ENSG00000091583,ENSMUSG00000000049,ortholog_one2one,100.0,1776.546401,2.975812e-03,4.951117,0.000003,0.2500,1.0,1,NA,protein_coding,protein_coding
4,ENSG00000141562,ENSMUSG00000000056,ortholog_one2one,100.0,70.950296,1.783816e-03,2.933593,0.661562,0.0000,1.0,1,NA,protein_coding,protein_coding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3855623,ENSG00000213024,ENSMUSG00000109511,ortholog_one2one,100.0,57.589531,1.531654e-03,3.701675,0.713207,0.0000,1.0,1,NA,protein_coding,protein_coding
3855628,ENSG00000183303,ENSMUSG00000109542,ortholog_one2many,75.0,0.027052,2.548884e-07,0.021907,1.172750,0.0000,1.0,21,Mouse,protein_coding,protein_coding
3855649,ENSG00000145700,ENSMUSG00000109561,ortholog_one2one,100.0,0.362324,3.574624e-06,0.440950,1.194783,0.0000,1.0,1,NA,protein_coding,protein_coding
3855650,ENSG00000181143,ENSMUSG00000109564,ortholog_one2one,75.0,11.742337,2.171618e-04,2.564331,0.237058,0.1250,1.0,1,NA,protein_coding,protein_coding


In [ ]:
for homologyType in dataProteinOnly["Mouse homology type"].unique():
    dataFiltered = dataProteinOnly[dataProteinOnly["Mouse homology type"] == homologyType].iloc[:, 4:9]
    print(f"N for {homologyType}: {dataFiltered.shape[0]}\n")
    print(f"Median for {homologyType}:\n{dataFiltered.median()}\n")
    print(f"Mean for {homologyType}:\n{dataFiltered.mean()}\n")

N for ortholog_one2one: 15889

Median for ortholog_one2one:
EuclidDist        46.725708
EuclidDistNorm     0.001135
EuclidDistLog      3.037357
PearDist           0.480553
TEC                0.062500
dtype: float64

Mean for ortholog_one2one:
EuclidDist        232.862199
EuclidDistNorm      0.003491
EuclidDistLog       3.432351
PearDist            0.525957
TEC                 0.072523
dtype: float64

N for ortholog_many2many: 3673

Median for ortholog_many2many:
EuclidDist        0.401128
EuclidDistNorm    0.000008
EuclidDistLog     0.107907
PearDist          1.077864
TEC               0.005747
dtype: float64

Mean for ortholog_many2many:
EuclidDist        475.991357
EuclidDistNorm      0.002879
EuclidDistLog       1.182786
PearDist            0.943316
TEC                 0.048836
dtype: float64

N for ortholog_one2many: 1605

Median for ortholog_one2many:
EuclidDist        20.301450
EuclidDistNorm     0.000257
EuclidDistLog      3.440866
PearDist           0.769553
TEC                

In [60]:
for species in dataProteinOnly["Duplicated Species"].unique()[1:]:
    dataFiltered = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species)].iloc[:, 4:9]
    print(f"N for {species}: {dataFiltered.shape[0]}\n")
    print(f"Median for {species}:\n{dataFiltered.median()}\n")
    print(f"Mean for {species}:\n{dataFiltered.mean()}\n")

N for Mouse: 1185

Median for Mouse:
EuclidDist        14.620565
EuclidDistNorm     0.000213
EuclidDistLog      3.536511
PearDist           0.823857
TEC                0.125000
dtype: float64

Mean for Mouse:
EuclidDist        322.581448
EuclidDistNorm      0.003025
EuclidDistLog       4.391775
PearDist            0.773956
TEC                 0.159124
dtype: float64

N for Human: 419

Median for Human:
EuclidDist        35.920691
EuclidDistNorm     0.000787
EuclidDistLog      3.365048
PearDist           0.694437
TEC                0.062500
dtype: float64

Mean for Human:
EuclidDist        1159.340851
EuclidDistNorm       0.007413
EuclidDistLog        4.268897
PearDist             0.685914
TEC                  0.180191
dtype: float64



In [62]:
for species in dataProteinOnly["Duplicated Species"].unique()[1:]:
    for gocScore in np.sort(dataProteinOnly["Mouse Gene-order conservation score"].unique()):
        dataFiltered = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species) & (dataProteinOnly["Mouse Gene-order conservation score"] == gocScore)].iloc[:, 4:9]
        print(f"N for {species} at GOC = {gocScore}: {dataFiltered.shape[0]}\n")
        print(f"Median for {species} at GOC = {gocScore}:\n{dataFiltered.median()}\n")
        print(f"Mean for {species} at GOC = {gocScore}:\n{dataFiltered.mean()}\n")

N for Mouse at GOC = 0.0: 511

Median for Mouse at GOC = 0.0:
EuclidDist        7.490131
EuclidDistNorm    0.000121
EuclidDistLog     1.945789
PearDist          0.855849
TEC               0.093750
dtype: float64

Mean for Mouse at GOC = 0.0:
EuclidDist        131.482271
EuclidDistNorm      0.001458
EuclidDistLog       4.064195
PearDist            0.837947
TEC                 0.157444
dtype: float64



N for Mouse at GOC = 25.0: 92

Median for Mouse at GOC = 25.0:
EuclidDist        7.934295
EuclidDistNorm    0.000114
EuclidDistLog     2.516506
PearDist          0.936193
TEC               0.110577
dtype: float64

Mean for Mouse at GOC = 25.0:
EuclidDist        1080.719361
EuclidDistNorm       0.007163
EuclidDistLog        3.053530
PearDist             0.803436
TEC                  0.182589
dtype: float64

N for Mouse at GOC = 50.0: 145

Median for Mouse at GOC = 50.0:
EuclidDist        9.452459
EuclidDistNorm    0.000154
EuclidDistLog     3.116542
PearDist          0.908171
TEC               0.145833
dtype: float64

Mean for Mouse at GOC = 50.0:
EuclidDist        104.142422
EuclidDistNorm      0.001376
EuclidDistLog       3.842522
PearDist            0.833576
TEC                 0.162914
dtype: float64

N for Mouse at GOC = 75.0: 86

Median for Mouse at GOC = 75.0:
EuclidDist        2.575588
EuclidDistNorm    0.000067
EuclidDistLog     1.313078
PearDist          0.635421
TEC          

In [68]:
for homologyType in np.sort(dataProteinOnly["Mouse homology type"].unique()):
    for species in dataProteinOnly["Duplicated Species"].unique()[1:]:
        groupOne = dataProteinOnly[dataProteinOnly["Mouse homology type"] == homologyType].iloc[:, 4:9]
        groupTwo = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species)].iloc[:, 4:9]
        print(f"P-Value for {homologyType} and {species}:\n{stats.mannwhitneyu(groupOne, groupTwo)[1]}\n")

P-Value for ortholog_many2many and Mouse:
[2.37955163e-122 7.12838592e-066 5.20573208e-170 9.73281990e-057
 5.39608793e-059]

P-Value for ortholog_many2many and Human:
[2.68065113e-098 1.55061076e-100 4.76910366e-108 1.84441027e-033
 8.75725201e-019]

P-Value for ortholog_one2many and Mouse:
[0.02597025 0.00310654 0.5923897  0.14466047 0.99638381]

P-Value for ortholog_one2many and Human:
[1.47957594e-05 7.47534228e-09 3.16653813e-01 4.91487006e-03
 9.96473454e-01]

P-Value for ortholog_one2one and Mouse:
[7.17787690e-060 3.54808770e-111 2.15228387e-001 1.51648518e-102
 1.02483870e-099]

P-Value for ortholog_one2one and Human:
[5.84202355e-03 5.36733743e-05 1.26735270e-02 5.99490806e-14
 4.18045993e-25]



In [64]:
for species in dataProteinOnly["Duplicated Species"].unique()[1:]:
    for gocScore in np.sort(dataProteinOnly["Mouse Gene-order conservation score"].unique()):
        if gocScore == 100:
            groupOne = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species) & (dataProteinOnly["Mouse Gene-order conservation score"] == 0)].iloc[:, 4:9]
            groupTwo = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species) & (dataProteinOnly["Mouse Gene-order conservation score"] == gocScore)].iloc[:, 4:9]
        else:                                                                                                                                                                                                               
            groupOne = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species) & (dataProteinOnly["Mouse Gene-order conservation score"] == gocScore)].iloc[:, 4:9]
            groupTwo = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == species) & (dataProteinOnly["Mouse Gene-order conservation score"] == gocScore + 25)].iloc[:, 4:9]
        print(f"P-Value for {species} and GOC = {gocScore}:\n{stats.mannwhitneyu(groupOne, groupTwo)[1]}\n")

P-Value for Mouse and GOC = 0.0:
[0.07917005 0.1204742  0.03751086 0.74841756 0.71734699]

P-Value for Mouse and GOC = 25.0:
[0.36628722 0.20579294 0.2215455  0.47594455 0.83327694]

P-Value for Mouse and GOC = 50.0:
[0.1011861  0.03722434 0.08831966 0.00086779 0.00264075]

P-Value for Mouse and GOC = 75.0:
[2.92619203e-12 1.42732057e-13 1.61650278e-09 3.37591093e-01
 5.23133197e-05]

P-Value for Mouse and GOC = 100.0:
[5.62342000e-22 1.07345898e-22 9.11442731e-10 3.84515236e-12
 1.42654969e-01]

P-Value for Human and GOC = 0.0:
[0.01091242 0.01073561 0.00844335 0.77682008 0.03029248]

P-Value for Human and GOC = 25.0:
[0.47171264 0.58264794 0.48049037 0.2266835  0.95970079]

P-Value for Human and GOC = 50.0:
[0.54613843 0.33510158 0.40215839 0.36290735 0.04929618]

P-Value for Human and GOC = 75.0:
[0.49495062 0.74924783 0.88779212 0.72319912 0.93862383]

P-Value for Human and GOC = 100.0:
[8.00833130e-01 9.10665645e-01 2.25781180e-04 2.26060269e-03
 8.26677403e-09]



In [70]:
groupOne = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == "Human")].iloc[:, 4:9]
groupTwo = dataProteinOnly[(dataProteinOnly["Mouse homology type"] == "ortholog_one2many") & (dataProteinOnly["Duplicated Species"] == "Mouse")].iloc[:, 4:9]
print(f"P-Value:\n{stats.mannwhitneyu(groupOne, groupTwo)[1]}\n")

P-Value:
[1.31261320e-08 3.60926165e-14 1.83910695e-01 2.18242268e-04
 9.98269166e-01]

